# Store customers, week 1 vs week 2: Pratt's method

**Wilcoxon Signed-Rank Test** · a worked example from the StatNex teaching guide

**The story.** Ten stores ran a two-week promotion. We recorded each store's average number of daily customers in week 1 and in week 2.

**The question.** Did stores get more customers in the second week of the promotion than in the first?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/wilcoxon_signed_rank.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Week 1,Week 2,Difference (Week 1 - Week 2)
1.0,182.0,194.0,-12.0
2.0,145.0,153.0,-8.0
3.0,210.0,207.0,3.0
4.0,96.0,111.0,-15.0
5.0,158.0,158.0,0.0
6.0,133.0,142.0,-9.0
7.0,240.0,261.0,-21.0
8.0,171.0,177.0,-6.0
9.0,119.0,114.0,5.0
10.0,205.0,216.0,-11.0

These cells save `signed_rank.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile signed_rank.py
"""The Wilcoxon signed-rank critical-value table, computed rather than looked up.

A printed table covers only the n somebody chose to publish; this covers whatever n
your data have. SciPy computes exact P values for a data set, but not the null
distribution itself, which is what a critical value is read from.
"""

import numpy as np
from scipy import stats

#: Past this many pairs printed tables stop, and the normal approximation stands in.
EXACT_LIMIT = 50


def signed_rank_null(n):
    """P(W = w) for w = 0 .. n(n+1)/2 when there are no ties.

    Each of the 2^n sign patterns is equally likely, so W is a sum over a random subset
    of the ranks 1..n: count the subsets by their sum, one rank at a time.
    """
    counts = np.zeros(n * (n + 1) // 2 + 1)
    counts[0] = 1.0
    for i in range(1, n + 1):
        np.add(counts[i:], counts[:-i].copy(), out=counts[i:])
    return counts / 2.0**n


def critical_w(n, alpha):
    """The largest W the table calls significant at tail probability ``alpha``.

    -1 when even W = 0 is not small enough: then nothing is significant.
    """
    if n > EXACT_LIMIT:
        mean = n * (n + 1) / 4
        sd = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        return float(np.floor(mean + stats.norm.ppf(alpha) * sd))
    eligible = np.flatnonzero(np.cumsum(signed_rank_null(n)) <= alpha)
    return float(eligible[-1]) if eligible.size else -1.0


def hodges_lehmann_interval(d, level=0.95):
    """The interval for the pseudo-median that R's wilcox.test(d, conf.int=TRUE) gives.

    R 4.6's rule, on every difference, zeros included. Under 50 differences it is exact.
    With no ties and no zeros it is the q-th smallest and q-th largest Walsh average
    (d_i + d_j)/2, i <= j, q the (1 - level)/2 point of the signed-rank sum. Otherwise the
    limits are the averages at which the exact tail probability of the signed-rank sum --
    over every sign pattern of the ranks actually observed at that shift -- first exceeds
    (1 - level)/2. Either way it covers at least the level. From 50 on, it is where the
    tie-corrected, continuity-corrected z crosses +-z, which happens at an average.

    Returns (lower, upper), both NaN when no interval reaches the level.
    """
    d = np.asarray(d, float)
    n = len(d)
    walsh = np.sort(np.add.outer(d, d)[np.triu_indices(n)]) / 2
    alpha = 1 - level
    toler = 10 * np.finfo(float).eps  # R's allowance for a tail equal to alpha/2
    if n < EXACT_LIMIT and len(np.unique(np.abs(d))) == n and not (d == 0).any():
        null = signed_rank_null(n)
        cumulative = np.cumsum(null)
        q = int(np.argmax(cumulative >= alpha / 2 - toler))  # R's qsignrank
        if cumulative[q] <= alpha / 2 + toler:
            q += 1
        if q == 0:
            return np.nan, np.nan
        return walsh[q - 1], walsh[len(walsh) - q]
    if n < EXACT_LIMIT:
        tables = {}

        def tail(mu, upper):
            # The exact tail of the signed-rank sum of d - mu, given its ties: the lower
            # tail P(V <= v), or the upper P(V > v), as R's interval code reads them.
            moved = d - mu
            doubled = np.rint(2 * stats.rankdata(np.abs(moved))).astype(int)
            key = tuple(sorted(doubled))
            if key not in tables:
                null = np.zeros(doubled.sum() + 1)
                null[0] = 1.0
                for rank in doubled:
                    shifted = np.zeros_like(null)
                    shifted[rank:] = null[:len(null) - rank]
                    null = 0.5 * null + 0.5 * shifted
                tables[key] = null
            null = tables[key]
            observed = int(doubled[moved > 0].sum())
            head = null[:observed + 1].sum()
            if not upper:
                return min(head, 1.0)
            return null[observed + 1:].sum() if abs(head - 1) < 1 / 64 else 1 - head

        limit = alpha / 2 + toler
        if tail(walsh[0] - 1, True) > limit or tail(walsh[-1] + 1, False) > limit:
            return np.nan, np.nan
        lower = walsh[-1]
        for k in range(len(walsh) - 1):
            if tail((walsh[k] + walsh[k + 1]) / 2, True) > limit:
                lower = walsh[k]
                break
        upper = walsh[0]
        for k in range(len(walsh) - 2, -1, -1):
            if tail((walsh[k] + walsh[k + 1]) / 2, False) > limit:
                upper = walsh[k + 1]
                break
        return lower, upper
    _, runs = np.unique(d, return_counts=True)
    tie_sum = (runs**3 - runs).sum()
    sd = np.sqrt(n * (n + 1) * (2 * n + 1) / 24 - tie_sum / 48)
    crit = stats.norm.ppf(1 - alpha / 2)

    def z_at(shift):
        # The continuity-corrected signed-rank z of d - shift, zeros dropped.
        moved = d - shift
        moved = moved[moved != 0]
        k = len(moved)
        ranks = stats.rankdata(np.abs(moved))
        _, ties = np.unique(ranks, return_counts=True)
        spread = k * (k + 1) * (2 * k + 1) / 24 - (ties**3 - ties).sum() / 48
        deviation = ranks[moved > 0].sum() - k * (k + 1) / 4
        return (deviation - np.sign(deviation) * 0.5) / np.sqrt(spread) if spread > 0 else np.nan

    # R first checks the ends of its range, the smallest and the largest difference: if
    # |z| does not pass the critical value even there, the level cannot be reached.
    ends = [z_at(d.min()), z_at(d.max())]
    if np.isnan(ends).any() or ends[0] < crit or ends[1] > -crit:
        return np.nan, np.nan
    values, counts = np.unique(walsh, return_counts=True)
    deviation = len(walsh) - np.cumsum(counts) - n * (n + 1) / 4  # V just above, less E
    z_steps = (deviation - np.sign(deviation) * 0.5) / sd
    return tuple(values[np.flatnonzero(z_steps <= c)[0]] if (z_steps <= c).any() else values[-1]
                 for c in (crit, -crit))

## Libraries

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats
from signed_rank import critical_w

pd.set_option("display.width", 120)

## The data

In [ ]:
# Two columns measured on the same cases. A pair with a blank or text cell is dropped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
first, second = "Week 1", "Week 2"
pairs = data[[first, second]].apply(pd.to_numeric, errors="coerce").dropna()
x = pairs[first].to_numpy(float)
y = pairs[second].to_numpy(float)
print(pd.DataFrame({"n": [len(x), len(y)]}, index=[first, second]))

## Paired differences

In [ ]:
# d = first - second, rounded to 12 significant digits of the data's scale. Binary
# floating point makes 1.1 - 1.0 and 0.3 - 0.2 differ in the 17th digit, although both
# are 0.1; without the rounding they would get different ranks instead of tying.
scale = np.abs(np.concatenate([x, y])).max()
d = x - y
if scale > 0:
    step = 10.0 ** (math.floor(math.log10(scale)) - 12)
    d = np.round(d / step) * step

## Case summary

In [ ]:
nonzero = d[d != 0]
zeros = int((d == 0).sum())
# Pratt (1959): the pairs with d = 0 take part in the ranking by |d| -- they are the
# smallest -- but carry no sign, so they count in neither rank sum. Ties share the
# average rank.
ranks = stats.rankdata(np.abs(d))[d != 0]
negative_sum = ranks[nonzero < 0].sum()
positive_sum = ranks[nonzero > 0].sum()
summary = pd.DataFrame(
    {"n": [zeros, int((nonzero < 0).sum()), int((nonzero > 0).sum()), len(d)],
     "rank sum": [np.nan, negative_sum, positive_sum, np.nan]},
    index=["d = 0", "d < 0", "d > 0", "total"],
)
print()
print(summary)

## Test by critical-value table

In [ ]:
n = len(nonzero)
# Two-sided: the statistic is the smaller rank sum, and each tail gets half of alpha.
statistic = min(negative_sum, positive_sum)
tail = {"1%": 0.01 / 2, "5%": 0.05 / 2}
# Significant when the statistic is at or below the table's critical value (no ties assumed).
verdict = next((f"significant at {level}" for level, alpha in tail.items()
                if statistic <= critical_w(n, alpha)), "not significant at 5%")
# With zeros in the ranking the statistic is on another scale than the table's, so
# only the normal approximation below judges it.
if zeros:
    verdict = "no table verdict: Pratt's ranking includes zero differences"
print(f"\nstatistic = {statistic}: {verdict}")

## Normal-approximation test

In [ ]:
# If nothing had changed, each difference would be as likely to be positive as
# negative, and each rank sum would sit near its expected value n(n + 1)/4. z says
# how many standard deviations the statistic is from it; P is read from the normal
# curve. The variance carries the tie correction sum(t^3 - t) / 48 over groups of tied
# |d|. scipy.stats.wilcoxon(d, zero_method="wilcox" or "pratt", correction=False,
# method="approx") gives the same z and P. It is written out here so that the ties
# are the ones the 12-digit rounding above decides, not ones floating-point error
# invents.
_, tie_sizes = np.unique(ranks, return_counts=True)
expected = n * (n + 1) / 4
variance = n * (n + 1) * (2 * n + 1) / 24 - (tie_sizes**3 - tie_sizes).sum() / 48
# Cureton (1967): with Pratt's ranking, the moments over all ranks less those the zeros
# hold.
if zeros:
    total = len(d)
    expected = (total * (total + 1) - zeros * (zeros + 1)) / 4
    variance = ((total * (total + 1) * (2 * total + 1) - zeros * (zeros + 1) * (2 * zeros + 1))
                / 24 - (tie_sizes**3 - tie_sizes).sum() / 48)
z = abs(statistic - expected) / np.sqrt(variance)
p_two = 2 * stats.norm.sf(z)
p = p_two
print(f"z = {z:.6f}, P = {p:.6g}")

## Exact test

In [ ]:
# Under the null hypothesis each difference is equally likely to be positive or negative,
# so W+ is the sum of a random half of the ranks actually observed -- ties, and Pratt's
# zeros, included. Doubled ranks are whole numbers, so the distribution is an array indexed
# by the doubled sum, built one rank at a time. (scipy.stats.wilcoxon(method="exact")
# assumes untied ranks 1..n and drops Pratt's zeros, so it cannot give this P.)
doubled = np.rint(2 * ranks).astype(int)
top = doubled.sum()
null = np.zeros(top + 1)
null[0] = 1.0
for rank in doubled:
    shifted = np.zeros_like(null)
    shifted[rank:] = null[: top + 1 - rank]
    null = 0.5 * null + 0.5 * shifted
observed = doubled[nonzero > 0].sum()
sums = np.arange(top + 1)
# Two-sided: W+ at least as far from its mean as observed.
exact_p = null[np.abs(2 * sums - top) >= abs(2 * observed - top)].sum()
exact_p = min(exact_p, 1.0)
print(f"exact P = {exact_p:.6g}")

## What the result says

The Wilcoxon signed-rank test ranks the differences between week 1 and week 2 by size, then adds the ranks of the gains and of the losses separately. One of the 10 stores has a difference of 0. Pratt's method keeps that store when it gives out the ranks, and only then drops its rank. So the other stores get higher ranks than when the zero is left out from the start.

The 7 stores with more customers in week 2 have a rank sum of 49, and the 2 stores with fewer have a rank sum of 5. The statistic is 5. The exact two-sided P value is 0.023, and the normal approximation gives z = 2.25 and P = 0.025. The P value is the chance of sums this uneven if week 2 were no different.

The answer does not depend on how the zero is handled: stores had more customers in week 2. The design cannot show that the promotion caused it.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.